In [45]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib





In [46]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Mini_Project/Car_details_v3.csv')

In [47]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Brand          2000 non-null   object
 1   year           2000 non-null   int64 
 2   km_driven      2000 non-null   int64 
 3   fuel           2000 non-null   object
 4   transmission   2000 non-null   object
 5   owner          2000 non-null   object
 6   mileage        1939 non-null   object
 7   engine         1939 non-null   object
 8   selling_price  2000 non-null   int64 
dtypes: int64(3), object(6)
memory usage: 140.8+ KB


In [48]:
df.sample(10)

,Brand,year,km_driven,fuel,transmission,owner,mileage,engine,selling_price
1454,Maruti,2016,100000,Diesel,Manual,1,24.3 kmpl,1248 CC,850000
406,Toyota,2016,140000,Diesel,Manual,1,12.99 kmpl,2494 CC,1200000
1341,Maruti,2007,35000,Petrol,Manual,1,18.2 kmpl,1061 CC,150000
468,Hyundai,2014,150360,Diesel,Manual,1,22.32 kmpl,1582 CC,645000
1483,Hyundai,2006,9558,Petrol,Manual,1,NaN,NaN,150000
453,Tata,2018,35000,Diesel,Manual,1,21.5 kmpl,1497 CC,700000
1072,Hyundai,2015,70000,Petrol,Automatic,2,18.9 kmpl,1197 CC,400000
195,Volkswagen,2011,65000,Diesel,Manual,2,20.54 kmpl,1598 CC,295000
1599,Tata,2011,85000,Diesel,Manual,2,18.0 kmpl,1248 CC,125000
1207,Ford,2017,76139,Diesel,Manual,1,23.0 kmpl,1498 CC,850000


In [49]:
df.describe()

,year,km_driven,selling_price
count,2000.000000,2.000000e+03,2.000000e+03
mean,2013.362500,7.453828e+04,5.095684e+05
std,4.041567,6.920939e+04,5.100045e+05
min,1996.000000,1.000000e+03,2.999900e+04
25%,2011.000000,4.000000e+04,2.500000e+05
50%,2014.000000,7.000000e+04,4.000000e+05
75%,2017.000000,1.000000e+05,6.250000e+05
max,2020.000000,2.360457e+06,7.200000e+06


# Preprocesing of Data




In [50]:
# mileage: "23.4 kmpl" / "17.3 km/kg" -> 23.4 / 17.3

df["mileage"] = df["mileage"].astype(str).str.extract(r"(\d+\.?\d*)").astype(float)

In [51]:
# engine: "1248 CC" -> 1248

df["engine"] = df["engine"].str.extract(r"(\d+)").astype(float)

In [52]:
# Handle missing values
df = df.dropna(subset=["mileage", "engine", "selling_price"])

In [53]:
# Drop duplicates

df = df.drop_duplicates()

In [54]:
# car_age instead of raw year (more stable feature)
CURRENT_YEAR = 2024
df["car_age"] = CURRENT_YEAR - df["year"]

In [55]:
# Drop raw year (we use car_age instead)
df = df.drop(columns=["year"])

In [56]:
# owner has '1'..'4' plus a few 'Test Drive Car' rows still left over —
df = df[df["owner"] != "Test Drive Car"]
df["owner"] = df["owner"].astype(int)

In [ ]:
df.tail()


,Brand,km_driven,fuel,transmission,owner,mileage,engine,selling_price,car_age
1995,Mahindra,170000,Diesel,Manual,2,13.60,2523.0,400000,15
1996,Hyundai,39000,Diesel,Manual,1,19.67,1582.0,819999,8
1997,Hyundai,25000,Petrol,Manual,1,21.10,814.0,310000,7
1998,Maruti,5000,Petrol,Automatic,2,20.51,998.0,445000,6
1999,Tata,10000,Petrol,Automatic,1,20.30,1199.0,645000,6


In [ ]:
df.to_csv("cleaned_car_data.csv", index=False)

In [57]:
# Split feature
categorical_cols = ["Brand", "fuel", "transmission"]
numeric_cols = ["km_driven", "owner", "mileage", "engine", "car_age"]

X = df[categorical_cols + numeric_cols]
y = df["selling_price"]

In [58]:
#Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [59]:
# Categorical encoding
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat = encoder.fit_transform(X_train[categorical_cols])
X_test_cat = encoder.transform(X_test[categorical_cols])

In [60]:
# Numeric scaling
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numeric_cols])
X_test_num = scaler.transform(X_test[numeric_cols])

X_train_final = np.hstack([X_train_cat, X_train_num])
X_test_final = np.hstack([X_test_cat, X_test_num])

In [61]:
# Train the model
model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_train_final, y_train)

RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)

In [62]:
# Evaluate
y_pred = model.predict(X_test_final)
print(f"R2 score : {r2_score(y_test, y_pred):.4f}")
print(f"MAE      : {mean_absolute_error(y_test, y_pred):,.0f}")
print(f"RMSE     : {np.sqrt(mean_squared_error(y_test, y_pred)):,.0f}")

R2 score : 0.7686
MAE      : 112,265
RMSE     : 238,021


In [63]:
#  Save artifacts
feature_columns = {
    "categorical_cols": categorical_cols,
    "numeric_cols": numeric_cols,
}
joblib.dump(model, "model.pkl")
joblib.dump(encoder, "encoders.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")
print("Saved: model.pkl, encoders.pkl, scaler.pkl, feature_columns.pkl")

Saved: model.pkl, encoders.pkl, scaler.pkl, feature_columns.pkl
